In [ ]:
import pandas as pd

# 1. 데이터 불러오기
df = pd.read_csv(
    "./data/safety_notice_processed.csv",
    encoding="utf-8-sig"
)

# 2. 작성일을 날짜형으로 변환
df["안전공지_작성일"] = pd.to_datetime(
    df["안전공지_작성일"],
    errors="coerce"
)

# 3. 연도 / 월 만들기
df["연도"] = df["안전공지_작성일"].dt.year.astype("Int64")
df["월"] = df["안전공지_작성일"].dt.month.astype("Int64")

# 4. 연도 + 월 + 대륙별 공지건수 집계
monthly_continent = (
    df.dropna(subset=["대륙명", "연도", "월"])
      .groupby(["연도", "월", "대륙명"])
      .size()
      .reset_index(name="공지건수")
)

display(monthly_continent.head(10))

In [ ]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# 1. 선택 위젯
# -------------------------

year_dropdown = widgets.Dropdown(
    options=["전체", 2025, 2026],
    value="전체",
    description="연도:"
)

continent_dropdown = widgets.Dropdown(
    options=["전체"] + sorted(
        monthly_continent["대륙명"].dropna().unique().tolist()
    ),
    value="전체",
    description="대륙:"
)


# -------------------------
# 2. 그래프 + 상세 데이터
# -------------------------

def draw_dashboard(year, continent):

    chart_df = monthly_continent.copy()

    # 연도 필터
    if year != "전체":
        chart_df = chart_df[
            chart_df["연도"] == year
        ]

    # 대륙 필터
    if continent != "전체":
        chart_df = chart_df[
            chart_df["대륙명"] == continent
        ]

    # 연월 표시
    chart_df["연월표시"] = chart_df.apply(
        lambda x: f"{str(x['연도'])[2:]}년 {x['월']}월",
        axis=1
    )

    # -------------------------
    # 그래프
    # -------------------------

    fig = px.line(
        chart_df,
        x="연월표시",
        y="공지건수",
        color="대륙명",
        markers=True,
        title="월별 대륙별 안전공지 추이"
    )

    fig.update_layout(
        xaxis_title="연월",
        yaxis_title="안전공지 건수",
        legend_title="대륙",
        hovermode="x unified"
    )

    fig.show()

    # -------------------------
    # 상세 데이터
    # -------------------------

    detail_df = df.copy()

    if year != "전체":
        detail_df = detail_df[
            detail_df["연도"] == year
        ]

    if continent != "전체":
        detail_df = detail_df[
            detail_df["대륙명"] == continent
        ]

# -------------------------
# 3. 대시보드 실행
# -------------------------

dashboard = widgets.interactive(
    draw_dashboard,
    year=year_dropdown,
    continent=continent_dropdown
)

display(dashboard)

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import pandas as pd


# ============================================================
# 1. 임베딩 모델
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
)


# ============================================================
# 2. 기존 Vector DB 불러오기
# ============================================================

vector_db = Chroma(
    persist_directory="data/vector_db",
    embedding_function=embeddings,
    collection_name="travel_safety",
)

print("Vector DB 저장 개수:", vector_db._collection.count())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector DB 저장 개수: 0


In [2]:
print(results[0].metadata)

NameError: name 'results' is not defined

In [3]:
import chromadb

client = chromadb.PersistentClient(
    path="data/vector_db"
)

collections = client.list_collections()

print("컬렉션 목록:", collections)

for collection in collections:
    print(
        "컬렉션 이름:",
        collection.name,
        "/ 저장 개수:",
        collection.count()
    )

컬렉션 목록: [Collection(name=travel_safety)]
컬렉션 이름: travel_safety / 저장 개수: 0


In [4]:
import chromadb

client = chromadb.PersistentClient(
    path="data/vector_db"
)

collection = client.get_collection("travel_safety")

print("count:", collection.count())

data = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

print(data)

count: 0
{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': []}


In [5]:
import chromadb
from pathlib import Path

db_path = Path(r"C:\Users\Playdata\Desktop\단위 프로젝트\mle-01-p1-team5\data\vector_db")
client = chromadb.PersistentClient(path=str(db_path))

for c in client.list_collections():
    print(c.name, c.count())

InternalError: Error executing plan: Error sending backfill request to compactor: Error constructing hnsw segment reader: Error creating hnsw segment reader: Error loading hnsw index